## 1. Loading and Inspecting the Dataset

This project examines animal shelter outcomes at the Austin Animal Center, focusing on dogs and cats from 2019 through 2024.

The dataset is published by the City of Austin and contains 173,775 records and 12 columns.

Pandas is used to load the CSV file, preview the first five records, and inspect the dataset's structure and data types. This initial inspection helps identify variables that may require cleaning before analysis.

In [1]:
import pandas as pd

# Load the Austin Animal Center dataset
df = pd.read_csv(
    "Austin_Animal_Center_Outcomes_(10_01_2013_to_05_05_2025)_20261115.csv",
    low_memory=False
)

# Check dataset dimensions
print("Dataset shape:", df.shape)

# Preview the first five rows
display(df.head())

# Examine column names and data types
df.info()

Dataset shape: (173775, 12)


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green
3,A679066,2014-04-16,NaN,2014-05-16T00:00:00-05:00,05-2014,NaN,NaN,Other,Unknown,4 weeks,Rabbit Sh,Brown
4,A680855,2014-05-25,NaN,2014-06-10T00:00:00-05:00,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Animal ID         173775 non-null  object
 1   Date of Birth     173775 non-null  object
 2   Name              123991 non-null  object
 3   DateTime          173775 non-null  object
 4   MonthYear         173775 non-null  object
 5   Outcome Type      173729 non-null  object
 6   Outcome Subtype   79660 non-null   object
 7   Animal Type       173775 non-null  object
 8   Sex upon Outcome  173774 non-null  object
 9   Age upon Outcome  173766 non-null  object
 10  Breed             173775 non-null  object
 11  Color             173775 non-null  object
dtypes: object(12)
memory usage: 15.9+ MB


## 2. Filtering the Dataset

The analysis focuses on dog and cat outcomes recorded between January 1, 2019, and December 31, 2024.

The DateTime column is converted into a usable date format, and the dataset is filtered by the selected years and animal types. A separate copy of the filtered data is created to preserve the original records.

The filtered dataset will be checked for record counts, animal-type distribution, and missing outcome information before additional cleaning.

In [2]:
# STEP 2: Filter the dataset for our project

# Extract the local calendar date from DateTime
df["Outcome Date"] = pd.to_datetime(
    df["DateTime"].str[:10],
    errors="coerce"
)

# Keep dogs and cats from 2019 through 2024
project_df = df[
    (df["Outcome Date"].between("2019-01-01", "2024-12-31")) &
    (df["Animal Type"].isin(["Dog", "Cat"]))
].copy()

# Verify the results
print("Filtered dataset shape:", project_df.shape)

print("\nAnimal types:")
print(project_df["Animal Type"].value_counts())

print("\nMissing outcome types:")
print(project_df["Outcome Type"].isna().sum())


Filtered dataset shape: (72192, 13)

Animal types:
Animal Type
Dog    39503
Cat    32689
Name: count, dtype: int64

Missing outcome types:
34


## 3. Initial Data Quality Assessment

Before cleaning the dataset, we examine missing values, duplicate records, and outcome categories.

This assessment helps identify potential data-quality issues and determine which records can be used reliably in the analysis.

Records will not be removed automatically. Any cleaning decisions will be documented and justified.

In [3]:
# STEP 3: Initial Data Quality Assessment

# Check missing values
print("Missing values by column:")
print(project_df.isna().sum())

# Check completely duplicated records
print("\nFully duplicated records:")
print(project_df.duplicated().sum())

# Examine outcome categories
print("\nOutcome type distribution:")
print(project_df["Outcome Type"].value_counts(dropna=False))

# Verify the date range
print("\nDate range:")
print("Earliest:", project_df["Outcome Date"].min())
print("Latest:", project_df["Outcome Date"].max())


Missing values by column:
Animal ID               0
Date of Birth           0
Name                17024
DateTime                0
MonthYear               0
Outcome Type           34
Outcome Subtype     39926
Animal Type             0
Sex upon Outcome        0
Age upon Outcome        0
Breed                   0
Color                   0
Outcome Date            0
dtype: int64

Fully duplicated records:
11

Outcome type distribution:
Outcome Type
Adoption           41162
Transfer           19407
Return to Owner     8253
Euthanasia          1472
Rto-Adopt            891
Died                 699
Disposal             225
Missing               37
NaN                   34
Relocate               6
Stolen                 5
Lost                   1
Name: count, dtype: int64

Date range:
Earliest: 2019-01-01 00:00:00
Latest: 2024-12-31 00:00:00


## 4. Investigating Duplicate Records

The initial data-quality assessment identified 11 fully duplicated records.

Before removing duplicates, the repeated records will be examined to determine whether they represent duplicate entries or potentially valid shelter encounters. This step helps prevent unnecessary data loss.

In [4]:
# STEP 4: Investigate duplicate records

duplicate_rows = project_df[
    project_df.duplicated(keep=False)
].sort_values(["Animal ID", "Outcome Date"])

print("Total rows involved in duplicate groups:", len(duplicate_rows))

print("\nDuplicate records:")
display(duplicate_rows[
    ["Animal ID", "Outcome Date", "Animal Type", "Outcome Type"]
])


Total rows involved in duplicate groups: 22

Duplicate records:


,Animal ID,Outcome Date,Animal Type,Outcome Type
94398,A783234,2019-01-13,Dog,Rto-Adopt
94399,A783234,2019-01-13,Dog,Rto-Adopt
102619,A797007,2019-06-09,Dog,Died
102620,A797007,2019-06-09,Dog,Died
114964,A811566,2020-01-09,Cat,Return to Owner
114965,A811566,2020-01-09,Cat,Return to Owner
118177,A815987,2020-04-13,Dog,Return to Owner
118178,A815987,2020-04-13,Dog,Return to Owner
123176,A826102,2020-11-23,Dog,Rto-Adopt
123177,A826102,2020-11-23,Dog,Rto-Adopt


## 5. Removing Exact Duplicate Records

The duplicate investigation identified 11 pairs of fully identical records.

One copy of each duplicate is removed to prevent repeated entries from affecting outcome counts and adoption percentages.

The original filtered dataset is preserved, while a separate cleaned DataFrame is created for further analysis. Records with the same Animal ID but different encounter information are retained.

In [5]:
# STEP 5: Remove exact duplicate records

# Preserve the filtered data and create a cleaned copy
project_clean = project_df.drop_duplicates().copy()

# Verify the cleaning results
print("Records before:", len(project_df))
print("Records after:", len(project_clean))

print("\nDuplicate records removed:",
      len(project_df) - len(project_clean))

print("\nRemaining exact duplicates:",
      project_clean.duplicated().sum())


Records before: 72192
Records after: 72181

Duplicate records removed: 11

Remaining exact duplicates: 0


## 6. Investigating Animal Ages

The Age upon Outcome column contains animal ages expressed in different units, including days, weeks, months, and years.

Before converting these values, we examine their distribution and identify potentially invalid or unusual entries.

This investigation will help us establish consistent age groups and document how missing or invalid age information is handled.

In [6]:
# STEP 6: Investigate animal ages

age_values = project_clean["Age upon Outcome"].astype("string").str.strip()

# Examine the most common age values
print("Most common age values:")
print(age_values.value_counts().head(20))

# Check for missing ages
print("\nMissing ages:", age_values.isna().sum())

# Identify negative ages
negative_ages = age_values.str.startswith("-", na=False)

print("\nRecords with negative ages:", negative_ages.sum())
display(project_clean.loc[
    negative_ages,
    ["Animal ID", "Animal Type", "Age upon Outcome", "Outcome Type"]
])

# Identify values with unexpected formats
valid_format = age_values.str.fullmatch(
    r"-?\d+\s+(?:day|days|week|weeks|month|months|year|years)",
    case=False,
    na=False
)

print("\nRecords with unexpected age formats:",
      (~valid_format).sum())

display(age_values[~valid_format].value_counts())


Most common age values:
Age upon Outcome
2 years      11865
2 months     11638
1 year        9535
3 months      4770
3 years       3570
1 month       3163
4 months      3002
4 years       2110
5 months      1958
5 years       1947
6 months      1745
2 weeks       1262
6 years       1223
3 weeks       1170
8 months      1127
7 years       1102
8 years       1031
7 months       995
10 months      938
4 weeks        933
Name: count, dtype: Int64

Missing ages: 0

Records with negative ages: 2


,Animal ID,Animal Type,Age upon Outcome,Outcome Type
96190,A725472,Dog,-1 years,Adoption
103006,A797495,Dog,-1 years,Return to Owner



Records with unexpected age formats: 0


Series([], Name: count, dtype: Int64)

## 7. Standardizing Ages and Creating Age Groups

Animal ages are converted into approximate years to ensure consistency across days, weeks, months, and years.

Four age groups are defined: under 1 year, 1–2 years, 3–6 years, and 7 years or older.

Two records contain invalid negative ages. These values are treated as missing for age-group comparisons, but the records are retained for other analyses.

This approach preserves useful data while preventing invalid ages from affecting the results.

In [7]:
# STEP 7: Standardize animal ages

# Extract age numbers and units
age_parts = (
    project_clean["Age upon Outcome"]
    .str.lower()
    .str.strip()
    .str.extract(r"^(-?\d+)\s+(day|week|month|year)s?$")
)

age_number = pd.to_numeric(age_parts[0], errors="coerce")

# Convert all ages into approximate years
multipliers = {
    "day": 1 / 365.25,
    "week": 7 / 365.25,
    "month": 1 / 12,
    "year": 1
}

project_clean["Age (Years)"] = (
    age_number * age_parts[1].map(multipliers)
)

# Treat invalid negative ages as missing
project_clean.loc[
    project_clean["Age (Years)"] < 0, "Age (Years)"
] = float("nan")

# Create four age groups
project_clean["Age Group"] = pd.cut(
    project_clean["Age (Years)"],
    bins=[0, 1, 3, 7, float("inf")],
    labels=["Under 1 year", "1–2 years",
            "3–6 years", "7+ years"],
    right=False
)

# Verify the results
print("Valid ages:",
      project_clean["Age (Years)"].notna().sum())

print("Missing/invalid ages:",
      project_clean["Age (Years)"].isna().sum())

print("\nAge group distribution:")
print(project_clean["Age Group"].value_counts(
    sort=False, dropna=False
))


Valid ages: 72179
Missing/invalid ages: 2

Age group distribution:
Age Group
Under 1 year    36577
1–2 years       21400
3–6 years        8850
7+ years         5352
NaN                 2
Name: count, dtype: int64


## 8. Preparing Outcome Categories

The Outcome Type column identifies how animals left the shelter.

Records with missing outcome types are excluded from adoption-share calculations because their outcomes are unknown.

The recorded category "Missing" is retained because it represents a documented outcome category rather than a blank value.

"Adoption" and "Rto-Adopt" are kept separate to avoid combining different outcome classifications without justification.

This preparation establishes a consistent denominator for subsequent adoption-percentage calculations.

In [8]:
# STEP 8: Prepare outcome categories

# Standardize outcome labels
project_clean["Outcome Type Clean"] = (
    project_clean["Outcome Type"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Keep records with a known outcome type
known_outcomes = project_clean[
    project_clean["Outcome Type Clean"].notna()
].copy()

# Verify the results
print("Total cleaned records:", len(project_clean))

print("Records with unknown outcomes:",
      project_clean["Outcome Type Clean"].isna().sum())

print("Records with known outcomes:", len(known_outcomes))

print("\nOutcome type distribution:")
print(known_outcomes["Outcome Type Clean"].value_counts())

print("\nRecords explicitly classified as Adoption:")
print((known_outcomes["Outcome Type Clean"] == "Adoption").sum())


Total cleaned records: 72181
Records with unknown outcomes: 34
Records with known outcomes: 72147

Outcome type distribution:
Outcome Type Clean
Adoption           41157
Transfer           19406
Return to Owner     8251
Euthanasia          1472
Rto-Adopt            889
Died                 698
Disposal             225
Missing               37
Relocate               6
Stolen                 5
Lost                   1
Name: count, dtype: Int64

Records explicitly classified as Adoption:
41157


## 9. Comparing Adoption Shares by Animal Type

This analysis compares the percentage of recorded outcomes classified as adoptions for dogs and cats from 2019 through 2024.

Adoption share is calculated by dividing the number of adoption outcomes by the total number of known outcomes for each animal type and multiplying by 100.

Both counts and percentages are reported to provide context. These percentages describe recorded shelter outcomes and should not be interpreted as an individual animal's probability of adoption.

In [9]:
# STEP 9: Compare adoption shares for dogs and cats

# Identify records classified as Adoption
known_outcomes["Is Adoption"] = (
    known_outcomes["Outcome Type Clean"] == "Adoption"
)

# Calculate adoption counts and known outcomes
species_summary = (
    known_outcomes
    .groupby("Animal Type")
    .agg(
        Known_Outcomes=("Is Adoption", "size"),
        Adoptions=("Is Adoption", "sum")
    )
    .reset_index()
)

# Calculate adoption percentage
species_summary["Adoption Share (%)"] = (
    species_summary["Adoptions"] /
    species_summary["Known_Outcomes"] * 100
).round(2)

# Display results
display(species_summary)

# Verify totals
print("Total known outcomes:",
      species_summary["Known_Outcomes"].sum())

print("Total adoption outcomes:",
      species_summary["Adoptions"].sum())


,Animal Type,Known_Outcomes,Adoptions,Adoption Share (%)
0,Cat,32676,19240,58.88
1,Dog,39471,21917,55.53


Total known outcomes: 72147
Total adoption outcomes: 41157


## 10. Annual Adoption Trends by Animal Type

This analysis examines how adoption shares changed between 2019 and 2024 for dogs and cats.

For each year and animal type, adoption share is calculated as the number of adoption outcomes divided by the number of known outcomes.

Comparing annual percentages will help identify trends, differences between animal types, and possible changes over time. These observations will form part of the project's data story.

In [10]:
# STEP 10: Calculate annual adoption shares

# Extract the year from the outcome date
known_outcomes["Year"] = known_outcomes["Outcome Date"].dt.year

# Group by year and animal type
annual_summary = (
    known_outcomes
    .groupby(["Year", "Animal Type"])
    .agg(
        Known_Outcomes=("Is Adoption", "size"),
        Adoptions=("Is Adoption", "sum")
    )
    .reset_index()
)

# Calculate annual adoption percentages
annual_summary["Adoption Share (%)"] = (
    annual_summary["Adoptions"] /
    annual_summary["Known_Outcomes"] * 100
).round(2)

# Display the annual results
display(annual_summary)

# Verify the totals
print("Total known outcomes:",
      annual_summary["Known_Outcomes"].sum())

print("Total adoptions:",
      annual_summary["Adoptions"].sum())


,Year,Animal Type,Known_Outcomes,Adoptions,Adoption Share (%)
0,2019,Cat,7766,3652,47.03
1,2019,Dog,11093,5629,50.74
2,2020,Cat,3522,1956,55.54
3,2020,Dog,5522,2800,50.71
4,2021,Cat,5094,3184,62.5
5,2021,Dog,6155,3543,57.56
6,2022,Cat,4764,3193,67.02
7,2022,Dog,6411,3410,53.19
8,2023,Cat,5521,3436,62.24
9,2023,Dog,5064,3075,60.72


Total known outcomes: 72147
Total adoptions: 41157


## 11. Visualizing Annual Adoption Trends

This interactive line chart compares annual adoption shares for dogs and cats from 2019 through 2024.

Both animal types experienced an increase in adoption share between the beginning and end of the study period, although the patterns were not consistent across every year.

The largest difference occurred in 2022, when cats had an adoption share 13.83 percentage points higher than dogs. By 2024, dogs had a slightly higher adoption share.

These comparisons describe the distribution of recorded shelter outcomes rather than the probability that an individual animal would be adopted.

In [11]:
# STEP 11: Interactive annual adoption trends

import plotly.express as px

fig = px.line(
    annual_summary.sort_values("Year"),
    x="Year",
    y="Adoption Share (%)",
    color="Animal Type",
    markers=True,
    title="Annual Adoption Share by Animal Type (2019–2024)",
    color_discrete_map={
        "Dog": "#3978A8",
        "Cat": "#D58E47"
    },
    hover_data={
        "Known_Outcomes": ":,",
        "Adoptions": ":,"
    }
)

# Improve readability and presentation
fig.update_layout(
    template="plotly_white",
    font=dict(size=15),
    height=520,
    legend_title_text="Animal Type",
    xaxis_title="Year",
    yaxis_title="Adoption Share (%)"
)

fig.update_xaxes(dtick=1)
fig.update_yaxes(range=[0, 100], ticksuffix="%")

fig.show()

# Save interactive HTML for Phase 2
fig.write_html("adoption_share_by_year.html")


## 12. Comparing Adoption Shares Across Age Groups

This analysis examines how adoption shares differ across four age groups for dogs and cats.

The age categories are Under 1 year, 1–2 years, 3–6 years, and 7+ years.

For each animal type and age group, adoption share is calculated using adoption outcomes divided by all known outcomes in that group.

Records with invalid ages are excluded only from this age-based comparison. Both outcome counts and percentages are examined to avoid misleading conclusions.

In [12]:
# STEP 12: Adoption shares by animal type and age group

# Exclude invalid ages only from the age analysis
age_valid = known_outcomes.dropna(
    subset=["Age Group"]
).copy()

# Calculate adoption counts by animal type and age group
age_summary = (
    age_valid
    .groupby(["Animal Type", "Age Group"], observed=True)
    .agg(
        Known_Outcomes=("Is Adoption", "size"),
        Adoptions=("Is Adoption", "sum")
    )
    .reset_index()
)

# Calculate adoption percentages
age_summary["Adoption Share (%)"] = (
    age_summary["Adoptions"] /
    age_summary["Known_Outcomes"] * 100
).round(2)

# Display results
display(age_summary)

# Verify totals
print("Records used in age analysis:", len(age_valid))

print("Records excluded due to invalid ages:",
      len(known_outcomes) - len(age_valid))


,Animal Type,Age Group,Known_Outcomes,Adoptions,Adoption Share (%)
0,Cat,Under 1 year,22030,14081,63.92
1,Cat,1–2 years,6876,3288,47.82
2,Cat,3–6 years,2234,1186,53.09
3,Cat,7+ years,1536,685,44.6
4,Dog,Under 1 year,14535,9483,65.24
5,Dog,1–2 years,14508,8034,55.38
6,Dog,3–6 years,6611,2980,45.08
7,Dog,7+ years,3815,1419,37.2


Records used in age analysis: 72145
Records excluded due to invalid ages: 2


## 13. Visualizing Adoption Shares by Age Group

This grouped bar chart compares adoption shares across four age categories for dogs and cats.

Animals younger than one year have the highest adoption shares for both species. The lowest shares occur among animals aged seven years or older.

The differences are especially noticeable for dogs, whose adoption share decreases across the four age groups. Cat adoption shares show a less consistent pattern.

These findings identify meaningful differences in recorded shelter outcomes without establishing that age causes those differences.

In [13]:
# STEP 13: Interactive adoption share by age group

age_order = [
    "Under 1 year",
    "1–2 years",
    "3–6 years",
    "7+ years"
]

fig_age = px.bar(
    age_summary,
    x="Age Group",
    y="Adoption Share (%)",
    color="Animal Type",
    barmode="group",
    title="Adoption Share by Age Group and Animal Type (2019–2024)",
    category_orders={"Age Group": age_order},
    color_discrete_map={
        "Dog": "#3978A8",
        "Cat": "#D58E47"
    },
    text="Adoption Share (%)",
    custom_data=["Known_Outcomes", "Adoptions"]
)

# Add readable percentage labels and hover details
fig_age.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    hovertemplate=(
        "Age group: %{x}<br>"
        "Adoption share: %{y:.2f}%<br>"
        "Known outcomes: %{customdata[0]:,}<br>"
        "Adoptions: %{customdata[1]:,}"
        "<extra>%{fullData.name}</extra>"
    )
)

# Improve layout
fig_age.update_layout(
    template="plotly_white",
    font=dict(size=15),
    height=520,
    xaxis_title="Age Group",
    yaxis_title="Adoption Share (%)",
    legend_title_text="Animal Type"
)

fig_age.update_yaxes(
    range=[0, 100],
    ticksuffix="%"
)

fig_age.show()

# Save the interactive visualization
fig_age.write_html("adoption_share_by_age.html")


## 14. Comparing Shelter Outcome Categories

This analysis examines how recorded shelter outcomes are distributed between dogs and cats from 2019 through 2024.

Outcome categories include Adoption, Transfer, Return to Owner, and other recorded classifications.

For each animal type, the percentage of outcomes in each category is calculated using the total number of known outcomes for that animal type.

This comparison provides additional context for the adoption-share findings without assuming that differences were caused by particular shelter practices.

In [14]:
# STEP 14: Investigate shelter outcome categories

# Count each outcome category by animal type
outcome_summary = (
    known_outcomes
    .groupby(["Animal Type", "Outcome Type Clean"])
    .size()
    .reset_index(name="Outcome Count")
)

# Calculate the total known outcomes for each animal type
outcome_summary["Total Outcomes"] = (
    outcome_summary
    .groupby("Animal Type")["Outcome Count"]
    .transform("sum")
)

# Calculate outcome percentages
outcome_summary["Outcome Share (%)"] = (
    outcome_summary["Outcome Count"] /
    outcome_summary["Total Outcomes"] * 100
).round(2)

# Display results
display(
    outcome_summary.sort_values(
        ["Animal Type", "Outcome Count"],
        ascending=[True, False]
    )
)

# Verify totals
print("Total outcomes:", outcome_summary["Outcome Count"].sum())

print("\nPercentage totals by animal type:")
print(
    outcome_summary.groupby("Animal Type")[
        "Outcome Share (%)"
    ].sum()
)


,Animal Type,Outcome Type Clean,Outcome Count,Total Outcomes,Outcome Share (%)
0,Cat,Adoption,19240,32676,58.88
8,Cat,Transfer,10615,32676,32.49
6,Cat,Return to Owner,1102,32676,3.37
3,Cat,Euthanasia,883,32676,2.70
1,Cat,Died,484,32676,1.48
7,Cat,Rto-Adopt,183,32676,0.56
2,Cat,Disposal,143,32676,0.44
4,Cat,Missing,20,32676,0.06
5,Cat,Relocate,6,32676,0.02
9,Dog,Adoption,21917,39471,55.53


Total outcomes: 72147

Percentage totals by animal type:
Animal Type
Cat    100.00
Dog     99.99
Name: Outcome Share (%), dtype: float64


## 15. Visualizing Shelter Outcome Distributions

This visualization compares the distribution of recorded outcomes for dogs and cats between 2019 and 2024.

The chart includes Adoption, Transfer, Return to Owner, and Other Recorded Outcomes. Less frequent categories are grouped for readability without altering their original classifications in the dataset.

Cats had a larger share of transfers, while dogs had a larger share of returns to owners. This provides important context beyond the adoption comparisons.

The findings describe recorded outcomes and do not establish the reasons for these differences.

In [15]:
# STEP 15: Interactive shelter outcome distribution

# Group smaller categories for visualization only
main_categories = [
    "Adoption",
    "Transfer",
    "Return to Owner"
]

outcome_chart_df = known_outcomes.copy()

outcome_chart_df["Outcome Category"] = (
    outcome_chart_df["Outcome Type Clean"]
    .where(
        outcome_chart_df["Outcome Type Clean"].isin(main_categories),
        "Other Recorded Outcomes"
    )
)

# Calculate counts and percentages
outcome_chart_summary = (
    outcome_chart_df
    .groupby(["Animal Type", "Outcome Category"])
    .size()
    .reset_index(name="Outcome Count")
)

outcome_chart_summary["Total Outcomes"] = (
    outcome_chart_summary
    .groupby("Animal Type")["Outcome Count"]
    .transform("sum")
)

outcome_chart_summary["Outcome Share (%)"] = (
    outcome_chart_summary["Outcome Count"] /
    outcome_chart_summary["Total Outcomes"] * 100
)

# Create the interactive chart
fig_outcomes = px.bar(
    outcome_chart_summary,
    x="Outcome Share (%)",
    y="Animal Type",
    color="Outcome Category",
    orientation="h",
    title="Distribution of Shelter Outcomes by Animal Type (2019–2024)",
    category_orders={
        "Animal Type": ["Dog", "Cat"],
        "Outcome Category": [
            "Adoption",
            "Transfer",
            "Return to Owner",
            "Other Recorded Outcomes"
        ]
    },
    color_discrete_map={
        "Adoption": "#2E7D32",
        "Transfer": "#7E57C2",
        "Return to Owner": "#E69F00",
        "Other Recorded Outcomes": "#9E9E9E"
    },
    custom_data=["Outcome Count", "Total Outcomes"]
)

# Add descriptive hover information
fig_outcomes.update_traces(
    hovertemplate=(
        "Share: %{x:.2f}%<br>"
        "Outcome count: %{customdata[0]:,}<br>"
        "Total known outcomes: %{customdata[1]:,}"
        "<extra>%{fullData.name}</extra>"
    )
)

# Improve readability
fig_outcomes.update_layout(
    template="plotly_white",
    barmode="stack",
    font=dict(size=15),
    height=450,
    xaxis_title="Share of Known Outcomes (%)",
    yaxis_title="Animal Type",
    legend_title_text="Outcome Category"
)

fig_outcomes.update_xaxes(
    range=[0, 100],
    dtick=20,
    ticksuffix="%"
)

fig_outcomes.show()

# Save interactive visualization
fig_outcomes.write_html("shelter_outcome_distribution.html")


## 16. Examining Annual Shelter Outcome Volume

This analysis examines the number of recorded dog and cat outcomes for each year from 2019 through 2024.

Annual outcome counts provide context for interpreting adoption percentages. A higher adoption share does not necessarily mean that more animals were adopted, particularly when the total number of recorded outcomes changes.

The analysis will help determine whether changes in adoption shares occurred alongside changes in shelter outcome volume.

In [16]:
# STEP 16: Examine annual shelter outcome volume

# Calculate annual known outcomes by animal type
annual_volume = (
    known_outcomes
    .groupby(["Year", "Animal Type"])
    .size()
    .reset_index(name="Known Outcomes")
)

# Display annual counts
display(annual_volume)

# Calculate total outcomes each year
yearly_totals = (
    annual_volume
    .groupby("Year")["Known Outcomes"]
    .sum()
    .reset_index()
)

print("Total known outcomes by year:")
display(yearly_totals)

# Verify overall total
print("Overall known outcomes:",
      yearly_totals["Known Outcomes"].sum())


,Year,Animal Type,Known Outcomes
0,2019,Cat,7766
1,2019,Dog,11093
2,2020,Cat,3522
3,2020,Dog,5522
4,2021,Cat,5094
5,2021,Dog,6155
6,2022,Cat,4764
7,2022,Dog,6411
8,2023,Cat,5521
9,2023,Dog,5064


Total known outcomes by year:


,Year,Known Outcomes
0,2019,18859
1,2020,9044
2,2021,11249
3,2022,11175
4,2023,10585
5,2024,11235


Overall known outcomes: 72147


## 17. Visualizing Annual Shelter Outcome Volume

This grouped bar chart compares the number of known dog and cat outcomes recorded each year from 2019 through 2024.

The number of recorded outcomes was substantially higher in 2019 than in 2024, even though adoption shares increased over the same period.

This distinction is important because a higher adoption percentage does not necessarily indicate a greater number of adoptions.

The chart provides context for interpreting adoption trends and avoids misleading conclusions based on percentages alone.

In [17]:
# STEP 17: Interactive annual outcome volume chart

fig_volume = px.bar(
    annual_volume,
    x="Year",
    y="Known Outcomes",
    color="Animal Type",
    barmode="group",
    title="Annual Shelter Outcome Volume by Animal Type (2019–2024)",
    color_discrete_map={
        "Dog": "#3978A8",
        "Cat": "#D58E47"
    },
    text="Known Outcomes"
)

# Improve labels and hover information
fig_volume.update_traces(
    texttemplate="%{text:,}",
    textposition="outside",
    hovertemplate=(
        "Year: %{x}<br>"
        "Known outcomes: %{y:,}"
        "<extra>%{fullData.name}</extra>"
    )
)

# Improve chart appearance
fig_volume.update_layout(
    template="plotly_white",
    font=dict(size=15),
    height=520,
    xaxis_title="Year",
    yaxis_title="Number of Known Outcome Records",
    legend_title_text="Animal Type",
    bargap=0.2
)

fig_volume.update_xaxes(dtick=1)
fig_volume.update_yaxes(rangemode="tozero")

fig_volume.show()

# Save the interactive HTML
fig_volume.write_html("annual_outcome_volume.html")


## 18. Preparing Data for the Streamlit Prototype

The cleaned dataset is exported to a separate CSV file for use in the Streamlit application.

Only variables needed for the interactive analysis are retained, including outcome date, animal type, outcome category, age group, and adoption classification.

This approach keeps the application organized and avoids repeating the full data-cleaning process. The original dataset and cleaning steps remain documented in the notebook.



Next, add this Python cell

In [18]:
# STEP 18: Export cleaned data for Streamlit

# Select columns required for the application
app_data = known_outcomes[
    [
        "Outcome Date",
        "Animal Type",
        "Outcome Type Clean",
        "Age Group",
        "Is Adoption"
    ]
].copy()

# Save the cleaned data
app_data.to_csv(
    "animal_shelter_clean.csv",
    index=False
)

# Verify the export
saved_data = pd.read_csv("animal_shelter_clean.csv")

print("Exported dataset shape:", saved_data.shape)

print("\nColumns:")
print(saved_data.columns.tolist())

print("\nAnimal types:")
print(saved_data["Animal Type"].value_counts())

print("\nAdoption outcomes:")
print(saved_data["Is Adoption"].sum())


Exported dataset shape: (72147, 5)

Columns:
['Outcome Date', 'Animal Type', 'Outcome Type Clean', 'Age Group', 'Is Adoption']

Animal types:
Animal Type
Dog    39471
Cat    32676
Name: count, dtype: int64

Adoption outcomes:
41157
